**ЗАДАНИЕ 1**

1. Обучить механизм сегментации изображений на наборе данных Pascal VOK. Или скачать с Яндекс.Диск

2. Случайным образом выбрать 100 изображений из тестовой выборки.

3. Выходной файл должен содержать 101 строку: На первой строке требуется записать средний IoU (по картинкам из выборки). Далее 100 строк должны иметь следующий формат:

<имя изображения> <средний IoU по объектам на изображении> <количество объектов на изображении> [<IoU между каждым результатом сегментирования и ground truth>]*

Решение должно быть реализовано в виде блокнота в формате .ipynb, ориентированного на запуск при помощи сервиса Google Colab с максимально подробными пояснениями реализации и интерпретации результатов. Все файлы, используемые в блокноте, должны скачиваться непосредственно в виртуальное окружение Google Colab.

Импортируем необходимые библиотеки:

In [2]:
import torch
torch.cuda.empty_cache()

In [3]:
# Установка необходимых пакетов (если их нет)
!pip install torch torchvision matplotlib scikit-image tqdm -q

import os
import random
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as T
from torchvision.transforms import functional as F
from torchvision.models.segmentation import deeplabv3_resnet101, DeepLabV3_ResNet101_Weights
from torchvision.datasets import VOCSegmentation
from tqdm.notebook import tqdm

# Фиксируем seed для воспроизводимости
random.seed(42)
torch.manual_seed(42)
np.random.seed(42)

print("Библиотеки загружены.")

Библиотеки загружены.


Загружаем набор данных Pascal VOC 2012, тренировачный и валидационный:

In [4]:
data_root = './data'
os.makedirs(data_root, exist_ok=True)

train_dataset_raw = VOCSegmentation(
    root=data_root, year='2012', image_set='train',
    download=True, transforms=None
)
val_dataset_raw = VOCSegmentation(
    root=data_root, year='2012', image_set='val',
    download=True, transforms=None
)

print(f"Тренировочных изображений: {len(train_dataset_raw)}")
print(f"Валидационных изображений: {len(val_dataset_raw)}")

100%|██████████| 2.00G/2.00G [01:32<00:00, 21.6MB/s]


Тренировочных изображений: 1464
Валидационных изображений: 1449


Для обучения и валидации мы приводим все изображения и маски к единому размеру (512×512). Изображения нормализуются (стандартные средние и стандартные отклонения для ImageNet), маски преобразуются в тензор целых чисел. При обучении дополнительно используется случайное горизонтальное отражение (аугментация).
Функция SimpleTransform будет применена к каждому элементу датасета.

In [5]:
import torch
import torchvision
from torchvision import datasets
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import random
from torchvision.transforms import functional as F
from torch.cuda.amp import autocast, GradScaler

DATA_DIR = './data'
IMAGE_SIZE = (512, 512)
BATCH_SIZE = 8
ACCUMULATION_STEPS = 2
NUM_EPOCHS = 10

class VOCSegmentationCustom(torch.utils.data.Dataset):
    def __init__(self, root, year, image_set, download=True):
        self.base_dataset = datasets.VOCSegmentation(
            root=root, year=year, image_set=image_set,
            download=download, transforms=None
        )
        self.is_train = (image_set == 'train')

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img = Image.open(self.base_dataset.images[idx]).convert('RGB')
        mask = Image.open(self.base_dataset.masks[idx])

        img = F.resize(img, IMAGE_SIZE, interpolation=F.InterpolationMode.BILINEAR)
        mask = F.resize(mask, IMAGE_SIZE, interpolation=F.InterpolationMode.NEAREST)

        if self.is_train and random.random() > 0.5:
            img = F.hflip(img)
            mask = F.hflip(mask)

        img = F.to_tensor(img)
        img = F.normalize(img, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        mask = torch.as_tensor(np.array(mask), dtype=torch.int64)

        return img, mask

train_dataset = VOCSegmentationCustom(root=DATA_DIR, year='2012', image_set='train', download=True)
val_dataset = VOCSegmentationCustom(root=DATA_DIR, year='2012', image_set='val', download=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

Используем предобученную модель deeplabv3_resnet101 с весами, полученными на датасете COCO. Обучаем модель на тренировочном наборе Pascal VOC. Количество эпох – 10.

In [6]:
from torchvision.models.segmentation import deeplabv3_resnet101, DeepLabV3_ResNet101_Weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
weights = DeepLabV3_ResNet101_Weights.DEFAULT
model = deeplabv3_resnet101(weights=weights)
model = model.to(device)

criterion = nn.CrossEntropyLoss(ignore_index=255)
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=0.0005)  # lr=0.001 вместо 0.01
scaler = torch.amp.GradScaler('cuda')

print(f"Начинаем проходить {NUM_EPOCHS} эпох... (размер {IMAGE_SIZE}, lr=0.001)")

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()  # один раз на эпоху, но градиенты будут накапливаться

    loop = tqdm(train_loader, desc=f"Эпоха {epoch+1}/{NUM_EPOCHS}")
    for i, (images, masks) in enumerate(loop):
        images, masks = images.to(device), masks.to(device)

        with torch.amp.autocast('cuda'):   # смешанная точность
            outputs = model(images)['out']
            loss = criterion(outputs, masks)

        scaler.scale(loss).backward()

        # градиентное накопление
        if (i + 1) % ACCUMULATION_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        running_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_loss = running_loss / len(train_loader)
    print(f"Эпоха {epoch+1} завершена. Средний loss: {avg_loss:.4f}")

torch.save(model.state_dict(), 'model_final.pth')
print("Модель сохранена")

Downloading: "https://download.pytorch.org/models/deeplabv3_resnet101_coco-586e9e4e.pth" to /root/.cache/torch/hub/checkpoints/deeplabv3_resnet101_coco-586e9e4e.pth


100%|██████████| 233M/233M [00:01<00:00, 127MB/s]


Начинаем проходить 10 эпох... (размер (512, 512), lr=0.001)


Эпоха 1/10: 100%|██████████| 183/183 [02:50<00:00,  1.07it/s, loss=0.124]


Эпоха 1 завершена. Средний loss: 0.1854


...

Эпоха 8 завершена. Средний loss: 0.0614


Эпоха 9/10: 100%|██████████| 183/183 [02:52<00:00,  1.06it/s, loss=0.0879]


Эпоха 9 завершена. Средний loss: 0.0565


Эпоха 10/10: 100%|██████████| 183/183 [02:52<00:00,  1.06it/s, loss=0.0354]


Эпоха 10 завершена. Средний loss: 0.0534
Модель сохранена


Для оценки качества нам потребуется функция, которая по предсказанной и истинной маске вычисляет IoU для каждого присутствующего класса (игнорируя фон). Результат – список IoU.

In [7]:
def compute_ious(pred_mask, true_mask):
    """
    pred_mask, true_mask: numpy массивы (H, W) с метками классов (0..20)
    Возвращает список IoU для классов, присутствующих в true_mask (исключая класс 0 – фон)
    """
    ious = []
    present_classes = np.unique(true_mask)
    present_classes = present_classes[present_classes != 0]
    for cls in present_classes:
        pred_bin = (pred_mask == cls)
        true_bin = (true_mask == cls)
        inter = np.logical_and(pred_bin, true_bin).sum()
        union = np.logical_or(pred_bin, true_bin).sum()
        iou = inter / union if union > 0 else 1.0
        ious.append(iou)
    return ious

Переключаем модель в режим оценки. Случайным образом выбираем 100 индексов из валидационного набора. Для каждого изображения получаем предсказание, вычисляем IoU для всех объектов и сохраняем данные.

In [8]:
model.eval()
num_samples = 100
indices = random.sample(range(len(val_dataset)), num_samples)

results = []  # (имя_файла, средний_IoU, число_объектов, список_IoU)

for idx in tqdm(indices, desc="Обработка изображений"):
    img, target = val_dataset[idx]
    # Получаем имя файла (без расширения)
    img_path = val_dataset_raw.images[idx]
    img_name = os.path.basename(img_path).replace('.jpg', '')

    # Предсказание
    img_tensor = img.unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(img_tensor)['out'][0]
    pred_mask = output.argmax(dim=0).cpu().numpy()
    true_mask = target.cpu().numpy()

    ious = compute_ious(pred_mask, true_mask)
    if not ious:
        # На изображении нет объектов – пропускаем
        continue

    mean_iou = np.mean(ious)
    num_objs = len(ious)
    results.append((img_name, mean_iou, num_objs, ious))

print(f"Обработано {len(results)} изображений из {num_samples}")

Обработка изображений: 100%|██████████| 100/100 [00:18<00:00,  5.47it/s]

Обработано 100 изображений из 100


Формируем файл results.txt

In [9]:
output_file = 'results.txt'
with open(output_file, 'w') as f:
    overall_mean = np.mean([r[1] for r in results])
    f.write(f"{overall_mean:.6f}\n")
    for name, mean_iou, num_objs, ious in results:
        iou_str = ' '.join(f"{x:.6f}" for x in ious)
        f.write(f"{name} {mean_iou:.6f} {num_objs} {iou_str}\n")

print(f"Результаты сохранены в {output_file}")
print(f"Общий средний IoU: {overall_mean:.4f}")

Результаты сохранены в results.txt
Общий средний IoU: 0.3894
